# Poster Figures: Postage Stamps and Difference Images

This notebook creates poster-friendly figures with very large text labels and captions.

Outputs include:
- Single-cluster postage stamp panels (Original, Injected, Difference)
- A full-frame Original / Injected / Difference panel
- High-resolution PNG files suitable for poster printing

In [ ]:
from pathlib import Path
import json

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm

try:
    from astropy.io import fits
except ImportError:
    fits = None

plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 300,
    'font.size': 20,
    'axes.titlesize': 24,
    'axes.labelsize': 20,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,
})

## 1) Configure Inputs (RSP-Aware)

This notebook is designed to reuse your RSP notebook outputs first.

Resolution order:
1. In-memory variables from your workflow (`image`, `injected_image`, `injection_info` or `catalog`)
2. Saved arrays from `outputs/multi_injection_rubin_psf/injected_image.npy`
3. Butler fallback (`dp02`, `deepCoadd`) to reconstruct the original image

You can still force direct file input if needed.

In [ ]:
# -------------------------
# USER SETTINGS
# -------------------------
# Prefer RSP notebook products first.
prefer_rsp_objects = True

# Candidate RSP output folders from your notebooks
rsp_output_dirs = [
    Path('../outputs/multi_injection_rubin_psf'),
    Path('../outputs/multi_injection_pipeline_with_diagnostics_rsp'),
    Path('../outputs/batch_run'),
]

# Butler fallback (used when `image` is not already available)
use_butler_fallback = True
butler_repo = 'dp02'
butler_collection = '2.2i/runs/DP0.2'
data_id = {'tract': 3828, 'patch': 24, 'band': 'i'}

# Optional direct file override (set use_direct_files=True to force)
use_direct_files = False
original_image_path = Path('../data/original_image.fits')
injected_image_path = Path('../data/injected_image.fits')

# Optional catalogs (if not available in-memory)
injection_catalog_path = Path('../tap_injection_results/injection_catalog.json')
detection_catalog_path = Path('../tap_injection_results/detections.json')

# Output folder for poster figures
output_dir = Path('../plots/poster')
output_dir.mkdir(parents=True, exist_ok=True)

# How many stamp examples to export
n_examples = 2
stamp_half_size = 28  # stamp size will be (2*half_size + 1) px

# If no catalog/injection_info is available, use manual coordinates
manual_positions = [
    (500, 500),
    (700, 820),
]

# Full-frame crop region for cleaner poster panel (set None for full image)
full_crop = None
# Example: full_crop = (200, 1400, 200, 1400)  # x0, x1, y0, y1

In [ ]:
def load_image(path: Path) -> np.ndarray:
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')

    suffix = path.suffix.lower()
    if suffix == '.npy':
        arr = np.load(path)
    elif suffix in {'.fits', '.fit', '.fz'}:
        if fits is None:
            raise ImportError('astropy is required to read FITS files. Install with: pip install astropy')
        with fits.open(path) as hdul:
            data_hdu = next((h for h in hdul if getattr(h, 'data', None) is not None), None)
            if data_hdu is None:
                raise ValueError(f'No image data found in FITS file: {path}')
            arr = np.asarray(data_hdu.data)
    else:
        raise ValueError(f'Unsupported image format: {path.suffix}')

    if arr.ndim != 2:
        raise ValueError(f'Expected 2D image, got shape {arr.shape} for {path}')

    return np.asarray(arr, dtype=float)


def robust_limits(arr: np.ndarray, lower=1, upper=99):
    finite = np.isfinite(arr)
    if not np.any(finite):
        return -1.0, 1.0
    vmin, vmax = np.percentile(arr[finite], [lower, upper])
    if np.isclose(vmin, vmax):
        vmax = vmin + 1.0
    return float(vmin), float(vmax)


def crop_or_full(arr: np.ndarray, crop=None):
    if crop is None:
        return arr
    x0, x1, y0, y1 = crop
    return arr[y0:y1, x0:x1]


def extract_stamp(arr: np.ndarray, x: float, y: float, half_size: int):
    xi = int(round(x))
    yi = int(round(y))

    y0 = max(0, yi - half_size)
    y1 = min(arr.shape[0], yi + half_size + 1)
    x0 = max(0, xi - half_size)
    x1 = min(arr.shape[1], xi + half_size + 1)
    return arr[y0:y1, x0:x1], (x0, x1, y0, y1)


def load_positions(catalog_path: Path):
    if not catalog_path.exists():
        return []

    if catalog_path.suffix.lower() == '.json':
        with open(catalog_path, 'r') as f:
            data = json.load(f)
        if isinstance(data, dict):
            if 'injection_info' in data and isinstance(data['injection_info'], list):
                data = data['injection_info']
            else:
                data = []
        pos = []
        for row in data:
            if isinstance(row, dict) and 'x' in row and 'y' in row:
                pos.append((float(row['x']), float(row['y'])))
        return pos

    if catalog_path.suffix.lower() == '.csv':
        import pandas as pd
        df = pd.read_csv(catalog_path)
        if not {'x', 'y'}.issubset(df.columns):
            return []
        return [(float(x), float(y)) for x, y in zip(df['x'].values, df['y'].values)]

    return []


def positions_from_records(records):
    pos = []
    for row in records:
        if isinstance(row, dict) and 'x' in row and 'y' in row:
            pos.append((float(row['x']), float(row['y'])))
    return pos

In [ ]:
original = None
injected = None
positions = []
detections = globals().get('detections', None)

# 1) In-memory RSP objects (if available in this notebook state)
if prefer_rsp_objects:
    g = globals()
    if 'image' in g and 'injected_image' in g:
        original = np.asarray(g['image'], dtype=float)
        injected = np.asarray(g['injected_image'], dtype=float)

    if 'injection_info' in g and isinstance(g['injection_info'], (list, tuple)):
        positions = positions_from_records(g['injection_info'])
    elif 'catalog' in g and isinstance(g['catalog'], (list, tuple)):
        positions = positions_from_records(g['catalog'])

# 2) Saved injected image from known output directories
if injected is None:
    for d in rsp_output_dirs:
        candidate = d / 'injected_image.npy'
        if candidate.exists():
            injected = load_image(candidate)
            print(f'Using injected image from: {candidate}')
            break

# 3) Butler fallback for original image
if original is None and use_butler_fallback:
    try:
        from lsst.daf.butler import Butler
        butler = Butler(butler_repo, collections=butler_collection)
        coadd = butler.get('deepCoadd', dataId=data_id)
        original = np.asarray(coadd.image.array, dtype=float)
        print('Loaded original image from Butler deepCoadd.')
    except Exception as e:
        print(f'Butler fallback failed: {e}')

# 4) Direct file override (if explicitly requested)
if use_direct_files:
    original = load_image(original_image_path)
    injected = load_image(injected_image_path)

# 5) Try loading detections from file if needed
if detections is None and detection_catalog_path.exists():
    if detection_catalog_path.suffix.lower() == '.json':
        with open(detection_catalog_path, 'r') as f:
            tmp = json.load(f)
        detections = tmp if isinstance(tmp, list) else []

if original is None or injected is None:
    checked = [str(d / 'injected_image.npy') for d in rsp_output_dirs]
    raise ValueError(
        'Could not resolve original/injected images.\n'
        'What was checked:\n'
        f'- In-memory vars: image={"image" in globals()}, injected_image={"injected_image" in globals()}\n'
        f'- Injected image files: {checked}\n'
        f'- Direct files mode: use_direct_files={use_direct_files}\n'
        '\nFix:\n'
        '1) Run your injection cells in the same kernel so `image` and `injected_image` exist, OR\n'
        '2) Re-run multi_injection_rubin_psf.ipynb Cell 4 to write outputs/.../injected_image.npy, OR\n'
        '3) Set use_direct_files=True and valid image paths.'
    )

if original.shape != injected.shape:
    raise ValueError(f'Image shape mismatch: original {original.shape}, injected {injected.shape}')

difference = injected - original

if len(positions) == 0:
    catalog_positions = load_positions(injection_catalog_path)
    positions = catalog_positions if len(catalog_positions) > 0 else manual_positions

if len(positions) == 0:
    raise ValueError('No stamp positions available. Provide injection_info/catalog/manual_positions.')

print(f'Loaded original shape: {original.shape}')
print(f'Loaded injected shape: {injected.shape}')
print(f'Using {len(positions)} candidate positions')
if detections is None:
    print('detections not found yet; completeness plots will wait until detections exist.')
else:
    print(f'detections loaded: {len(detections)}')

## 2) Full-Frame Poster Panel (Original, Injected, Difference)

In [ ]:
orig_show = crop_or_full(original, full_crop)
inj_show = crop_or_full(injected, full_crop)
diff_show = crop_or_full(difference, full_crop)

ovmin, ovmax = robust_limits(np.concatenate([orig_show.ravel(), inj_show.ravel()]), 1, 99)
dv = np.nanpercentile(np.abs(diff_show), 99.5)
dv = max(dv, 1e-6)

fig, axes = plt.subplots(1, 3, figsize=(24, 8), constrained_layout=True)

im0 = axes[0].imshow(orig_show, origin='lower', cmap='gray', vmin=ovmin, vmax=ovmax)
axes[0].set_title('Original Image', fontweight='bold')
axes[0].set_xlabel('X [pix]', fontweight='bold')
axes[0].set_ylabel('Y [pix]', fontweight='bold')
cb0 = fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02)
cb0.set_label('Flux', fontweight='bold')

im1 = axes[1].imshow(inj_show, origin='lower', cmap='gray', vmin=ovmin, vmax=ovmax)
axes[1].set_title('Injected Image', fontweight='bold')
axes[1].set_xlabel('X [pix]', fontweight='bold')
axes[1].set_ylabel('Y [pix]', fontweight='bold')
cb1 = fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02)
cb1.set_label('Flux', fontweight='bold')

im2 = axes[2].imshow(
    diff_show,
    origin='lower',
    cmap='coolwarm',
    norm=SymLogNorm(linthresh=max(dv/30, 1e-6), vmin=-dv, vmax=dv)
)
axes[2].set_title('Injected - Original', fontweight='bold')
axes[2].set_xlabel('X [pix]', fontweight='bold')
axes[2].set_ylabel('Y [pix]', fontweight='bold')
cb2 = fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.02)
cb2.set_label('Delta Flux', fontweight='bold')

fig.suptitle('Cluster Injection Diagnostics', fontsize=30, fontweight='bold')
out = output_dir / 'poster_fullframe_original_injected_difference.png'
fig.savefig(out, dpi=350, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## 3) A Couple of Single Postage-Stamp Panels

This cell exports one large figure per selected source with clear labels and a bold caption.

In [ ]:
def make_stamp_triptych(
    original_img, injected_img, x, y, half_size=28, idx=0, output_dir=Path('.'), show=True
):
    diff_img = injected_img - original_img

    o_stamp, _ = extract_stamp(original_img, x, y, half_size)
    i_stamp, _ = extract_stamp(injected_img, x, y, half_size)
    d_stamp, _ = extract_stamp(diff_img, x, y, half_size)

    svmin, svmax = robust_limits(np.concatenate([o_stamp.ravel(), i_stamp.ravel()]), 1, 99)
    dmax = np.nanpercentile(np.abs(d_stamp), 99.5)
    dmax = max(dmax, 1e-6)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

    im0 = axes[0].imshow(o_stamp, origin='lower', cmap='gray', vmin=svmin, vmax=svmax)
    axes[0].set_title('Original Stamp', fontsize=22, fontweight='bold')
    axes[0].set_xlabel('X [pix]', fontsize=18, fontweight='bold')
    axes[0].set_ylabel('Y [pix]', fontsize=18, fontweight='bold')
    cb0 = fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02)
    cb0.set_label('Flux', fontsize=16, fontweight='bold')

    im1 = axes[1].imshow(i_stamp, origin='lower', cmap='gray', vmin=svmin, vmax=svmax)
    axes[1].set_title('Injected Stamp', fontsize=22, fontweight='bold')
    axes[1].set_xlabel('X [pix]', fontsize=18, fontweight='bold')
    axes[1].set_ylabel('Y [pix]', fontsize=18, fontweight='bold')
    cb1 = fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02)
    cb1.set_label('Flux', fontsize=16, fontweight='bold')

    im2 = axes[2].imshow(
        d_stamp,
        origin='lower',
        cmap='coolwarm',
        norm=SymLogNorm(linthresh=max(dmax / 30, 1e-6), vmin=-dmax, vmax=dmax)
    )
    axes[2].set_title('Difference (Injected - Original)', fontsize=22, fontweight='bold')
    axes[2].set_xlabel('X [pix]', fontsize=18, fontweight='bold')
    axes[2].set_ylabel('Y [pix]', fontsize=18, fontweight='bold')
    cb2 = fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.02)
    cb2.set_label('Delta Flux', fontsize=16, fontweight='bold')

    caption = f'Source {idx + 1}: (x, y) = ({x:.1f}, {y:.1f}), stamp size = {2 * half_size + 1} px'
    fig.suptitle(caption, fontsize=26, fontweight='bold')

    out = output_dir / f'poster_stamp_triptych_{idx + 1:02d}.png'
    fig.savefig(out, dpi=400, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)

    return out


selected = positions[:n_examples]
saved_files = []
for idx, (x, y) in enumerate(selected):
    out = make_stamp_triptych(
        original,
        injected,
        x=float(x),
        y=float(y),
        half_size=stamp_half_size,
        idx=idx,
        output_dir=output_dir,
        show=True,
    )
    saved_files.append(out)

print('Saved stamp panels:')
for f in saved_files:
    print(' -', f)

## 4) Optional: One Combined Summary Figure for Exactly Two Examples

If `n_examples >= 2`, this cell creates one poster panel containing two rows (one source per row).

In [ ]:
if len(positions) >= 2:
    pair = positions[:2]
    fig, axes = plt.subplots(2, 3, figsize=(20, 12), constrained_layout=True)

    for row, (x, y) in enumerate(pair):
        o_stamp, _ = extract_stamp(original, x, y, stamp_half_size)
        i_stamp, _ = extract_stamp(injected, x, y, stamp_half_size)
        d_stamp = i_stamp - o_stamp

        vmin, vmax = robust_limits(np.concatenate([o_stamp.ravel(), i_stamp.ravel()]), 1, 99)
        dmax = max(np.nanpercentile(np.abs(d_stamp), 99.5), 1e-6)

        ims = [
            axes[row, 0].imshow(o_stamp, origin='lower', cmap='gray', vmin=vmin, vmax=vmax),
            axes[row, 1].imshow(i_stamp, origin='lower', cmap='gray', vmin=vmin, vmax=vmax),
            axes[row, 2].imshow(
                d_stamp, origin='lower', cmap='coolwarm',
                norm=SymLogNorm(linthresh=max(dmax/30, 1e-6), vmin=-dmax, vmax=dmax)
            ),
        ]

        titles = ['Original', 'Injected', 'Difference']
        for col in range(3):
            axes[row, col].set_title(titles[col], fontsize=20, fontweight='bold')
            axes[row, col].set_xlabel('X [pix]', fontsize=16, fontweight='bold')
            axes[row, col].set_ylabel('Y [pix]', fontsize=16, fontweight='bold')
            fig.colorbar(ims[col], ax=axes[row, col], fraction=0.046, pad=0.02)

        axes[row, 0].text(
            0.02, 0.98,
            f'Source {row + 1}: (x, y)=({x:.1f}, {y:.1f})',
            transform=axes[row, 0].transAxes,
            va='top', ha='left',
            fontsize=16, fontweight='bold',
            color='white',
            bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', pad=6),
        )

    fig.suptitle('Poster Stamp Comparison: Two Example Injections', fontsize=30, fontweight='bold')
    combined_out = output_dir / 'poster_two_example_stamps_grid.png'
    fig.savefig(combined_out, dpi=400, bbox_inches='tight')
    plt.show()
    print(f'Saved: {combined_out}')
else:
    print('Need at least 2 positions for this panel.')

## 5) Completeness Plots (Same Pattern as Your RSP Notebooks)

This section follows the same workflow used in your pipeline notebooks:
- `ClusterRetrieval(injection_info, detections)`
- `match_detections(match_radius=5.0)`
- `compute_completeness(parameter='magnitude'/'r_half', bins=...)`
- optional 2D completeness map

In [ ]:
from src.retrieval import ClusterRetrieval

# Resolve injection_info from current kernel or fallback catalog
if 'injection_info' in globals() and isinstance(injection_info, (list, tuple)):
    inj_for_retrieval = list(injection_info)
elif 'catalog' in globals() and isinstance(catalog, (list, tuple)):
    inj_for_retrieval = list(catalog)
else:
    # Fallback to JSON catalog if available
    if injection_catalog_path.exists():
        with open(injection_catalog_path, 'r') as f:
            tmp = json.load(f)
        inj_for_retrieval = tmp if isinstance(tmp, list) else []
    else:
        inj_for_retrieval = []

if detections is None:
    detections = []

if len(inj_for_retrieval) == 0:
    raise ValueError('No injection_info/catalog available for completeness analysis.')

if len(detections) == 0:
    print('No detections provided. Run your detection pipeline first, then re-run this cell.')
else:
    retrieval = ClusterRetrieval(inj_for_retrieval, detections)
    retrieval.match_detections(match_radius=5.0)
    stats = retrieval.get_summary_statistics()

    print('=== Completeness Results ===')
    print(f"Injected clusters: {stats['n_injected']}")
    print(f"Recovered clusters: {stats['n_detected']}")
    print(f"Completeness: {stats['overall_completeness']:.1%}")

    # Completeness vs magnitude (same pattern as RSP notebook)
    mag_bins = np.linspace(20, 26, 13)
    bc_mag, comp_mag, comp_err_mag = retrieval.compute_completeness(parameter='magnitude', bins=mag_bins)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.step(bc_mag, comp_mag, where='mid', lw=2, color='royalblue', label='Completeness')
    ax.fill_between(bc_mag, comp_mag - comp_err_mag, comp_mag + comp_err_mag,
                    step='mid', alpha=0.25, color='royalblue', label='1-sigma')
    ax.set_xlabel('Magnitude')
    ax.set_ylabel('Completeness')
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)
    ax.set_title('Completeness vs Magnitude')
    ax.legend()
    out_mag = output_dir / 'completeness_vs_mag.png'
    plt.savefig(out_mag, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_mag}')

    # Completeness vs r_half (same pattern as RSP notebook)
    rh_bins = np.linspace(2, 12, 11)
    bc_rh, comp_rh, comp_err_rh = retrieval.compute_completeness(parameter='r_half', bins=rh_bins)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.step(bc_rh, comp_rh, where='mid', lw=2, color='darkorange', label='Completeness')
    ax.fill_between(bc_rh, comp_rh - comp_err_rh, comp_rh + comp_err_rh,
                    step='mid', alpha=0.25, color='darkorange', label='1-sigma')
    ax.set_xlabel('r_half (px)')
    ax.set_ylabel('Completeness')
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)
    ax.set_title('Completeness vs r_half')
    ax.legend()
    out_rh = output_dir / 'completeness_vs_rh.png'
    plt.savefig(out_rh, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_rh}')

    # Optional 2D map
    mag_bins2 = np.linspace(20, 26, 13)
    rh_bins2 = np.linspace(2, 12, 11)
    result2d = retrieval.compute_completeness_2d('magnitude', 'r_half', mag_bins2, rh_bins2)

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.pcolormesh(mag_bins2, rh_bins2, result2d['completeness'].T,
                       cmap='viridis', vmin=0, vmax=1, shading='auto')
    plt.colorbar(im, ax=ax, label='Completeness')
    ax.set_xlabel('Magnitude')
    ax.set_ylabel('r_half (px)')
    ax.set_title('2D Completeness: Magnitude x r_half')
    out_2d = output_dir / 'completeness_2d.png'
    plt.savefig(out_2d, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_2d}')